In [13]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import pandas as pd

# 데이터 불러오기
url = "https://raw.githubusercontent.com/sehakflower/data/main/titanic.csv"
data = pd.read_csv(url, sep="\t")

# 데이터 기본 정보 확인
print(data.head())
print(data.info())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

In [14]:
# Name 열에서 호칭(title) 추출
data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# 대표적인 호칭 그룹화
data['Title'] = data['Title'].replace(['Mlle','Ms'], 'Miss')
data['Title'] = data['Title'].replace(['Mme'], 'Mrs')
data['Title'] = data['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
    'Other'
)

# 숫자형 변환
title_mapping = {'Mr':0, 'Miss':1, 'Mrs':2, 'Master':3, 'Other':4}
data['Title'] = data['Title'].map(title_mapping)

# 확인
print(data[['Name','Title']].head())


                                                Name  Title
0                            Braund, Mr. Owen Harris      0
1  Cumings, Mrs. John Bradley (Florence Briggs Th...      2
2                             Heikkinen, Miss. Laina      1
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)      2
4                           Allen, Mr. William Henry      0


In [15]:
# 호칭별 중앙값 계산
title_age_median = data.groupby('Title')['Age'].median()

# Age 결측치 채우기
for title, median in title_age_median.items():
    data.loc[(data['Age'].isnull()) & (data['Title'] == title), 'Age'] = median

# 확인
print(data[['Title','Age']].head(20))
print(data.isnull().sum())


    Title   Age
0       0  22.0
1       2  38.0
2       1  26.0
3       2  35.0
4       0  35.0
5       0  28.0
6       0  54.0
7       3   2.0
8       2  27.0
9       2  14.0
10      1   4.0
11      1  58.0
12      0  20.0
13      0  39.0
14      1  14.0
15      2  55.0
16      3   2.0
17      0  28.0
18      2  31.0
19      2  31.0
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          125
Embarked         1
Title            0
dtype: int64


In [16]:
# Sex 열 인코딩: male=0, female=1
sex_mapping = {'male':0, 'female':1}
data['Sex'] = data['Sex'].map(sex_mapping)

# 확인
print(data[['Sex']].head())


   Sex
0    0
1    1
2    1
3    1
4    0


In [17]:
# Embarked 결측치 처리: 최빈값으로 채우기
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)

# Embarked 인코딩: C=0, Q=1, S=2
embarked_mapping = {'C':0, 'Q':1, 'S':2}
data['Embarked'] = data['Embarked'].map(embarked_mapping)

# 확인
print(data[['Embarked']].head())
print(data['Embarked'].value_counts())


   Embarked
0         2
1         0
2         2
3         2
4         2
Embarked
2    111
0     32
1     13
Name: count, dtype: int64


/tmp/ipykernel_382/1099728195.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)


In [18]:
# SibSp, Parch 열 확인
print(data[['SibSp','Parch']].head())


   SibSp  Parch
0      1      0
1      1      0
2      0      0
3      1      0
4      0      0


In [19]:
# 개인별 요금 계산: Fare ÷ (SibSp + Parch + 1)
data['fare_person'] = data['Fare'] / (data['SibSp'] + data['Parch'] + 1)

# 확인
print(data[['Fare','SibSp','Parch','fare_person']].head())


      Fare  SibSp  Parch  fare_person
0   7.2500      1      0      3.62500
1  71.2833      1      0     35.64165
2   7.9250      0      0      7.92500
3  53.1000      1      0     26.55000
4   8.0500      0      0      8.05000


In [21]:
# 최종 데이터셋 만들기 (불필요한 열 제거)
data_final = data.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)

print(data_final.head())
print(data_final.info())


   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  Title  \
0         0       3    0  22.0      1      0   7.2500         2      0   
1         1       1    1  38.0      1      0  71.2833         0      2   
2         1       3    1  26.0      0      0   7.9250         2      1   
3         1       1    1  35.0      1      0  53.1000         2      2   
4         0       3    0  35.0      0      0   8.0500         2      0   

   fare_person  
0      3.62500  
1     35.64165  
2      7.92500  
3     26.55000  
4      8.05000  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     156 non-null    int64  
 1   Pclass       156 non-null    int64  
 2   Sex          156 non-null    int64  
 3   Age          156 non-null    float64
 4   SibSp        156 non-null    int64  
 5   Parch        156 non-null    int64  
 6   Fare   

In [22]:
#DecisionTreeClassifier적용
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = dt_model.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred)

print("DecisionTreeClassifier 정확도:", acc_dt)


DecisionTreeClassifier 정확도: 0.78125


In [23]:
#RandomFaorestClassifier 적용
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred)

print("RandomForestClassifier 정확도:", acc_rf)


RandomForestClassifier 정확도: 0.78125


In [24]:
#LogistictRegression 적용
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = lr_model.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred)

print("LogisticRegression 정확도:", acc_lr)


LogisticRegression 정확도: 0.78125


In [25]:
#GaussianNB적
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = nb_model.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred)

print("GaussianNB 정확도:", acc_nb)


GaussianNB 정확도: 0.71875


In [26]:
#KNeighborsClassifier 적용
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습 (기본값: n_neighbors=5)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = knn_model.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred)

print("KNeighborsClassifier 정확도:", acc_knn)


KNeighborsClassifier 정확도: 0.71875


In [27]:
#LinearSVC적용
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
svc_model = LinearSVC(max_iter=10000, random_state=42)
svc_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = svc_model.predict(X_test)
acc_svc = accuracy_score(y_test, y_pred)

print("LinearSVC 정확도:", acc_svc)


LinearSVC 정확도: 0.71875


In [28]:
#Perceptron 적용
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
perceptron_model = Perceptron(random_state=42)
perceptron_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = perceptron_model.predict(X_test)
acc_perceptron = accuracy_score(y_test, y_pred)

print("Perceptron 정확도:", acc_perceptron)



Perceptron 정확도: 0.75


In [29]:
#SGDClassifier적용
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 데이터 분할
X = data_final.drop('Survived', axis=1)
y = data_final['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
sgd_model = SGDClassifier(random_state=42)
sgd_model.fit(X_train, y_train)

# 예측 및 정확도 평가
y_pred = sgd_model.predict(X_test)
acc_sgd = accuracy_score(y_test, y_pred)

print("SGDClassifier 정확도:", acc_sgd)



SGDClassifier 정확도: 0.65625


In [ ]:
# DecisionTreeClassifier, ForestClassifier, LogisticRegression 이 세가지가 높게나옴.
